[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/projects/05_recommendation/project.ipynb)

# Project 05 — Gợi ý phim (Recommender System)

> Một ứng dụng xem phim muốn hiển thị **10 phim gợi ý** trên trang chủ của mỗi người dùng. Hiện tại trang chủ hiện 10 phim **phổ biến nhất** cho mọi người. Dữ liệu: 5.441 lượt chấm điểm (1–5 sao, có thời điểm) của 300 người dùng cho 200 phim (`ratings.csv`), và thể loại phim (`items.csv`).

**Bạn sẽ làm**:
1. EDA: phân phối đuôi dài, thiên lệch chọn lọc của điểm số.
2. Chia **theo thời gian cho từng người dùng** (20% lượt gần nhất làm test); định nghĩa "phim thích" = điểm ≥ 4.
3. Hàm đánh giá top-10: Precision@10, Recall@10, NDCG@10, độ phủ danh mục.
4. Ba phương pháp: phổ biến, item-kNN (cosine), PureSVD (chọn số chiều bằng validation); thêm gợi ý theo thể loại cho người dùng mới.
5. Phân tích thiên lệch phổ biến; báo cáo.

**Tiêu chí đạt**: item-kNN có NDCG@10 cao hơn phổ biến ít nhất 15% · PureSVD (số chiều chọn trên validation) có NDCG@10 cao hơn phổ biến và độ phủ danh mục gấp ≥ 2 lần.

**Chương liên quan**: 26 (gợi ý, CF, matrix factorization, metric xếp hạng), 21 (SVD), 14 (metric), 15 (chia theo thời gian).

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/projects/05_recommendation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse.linalg import svds

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

ratings = pd.read_csv(repo_path("data", "sample", "ratings.csv"))
items = pd.read_csv(repo_path("data", "sample", "items.csv"))
N_USERS, N_ITEMS = ratings.user_id.max() + 1, items.item_id.max() + 1
print(ratings.shape, f"{ratings.user_id.nunique()} người dùng, {ratings.item_id.nunique()} phim")
ratings.head()

## Bước 1 — EDA

Tính: tỉ lệ lượt chấm của 10% phim phổ biến nhất `top10_share`; phân phối điểm `rating_dist` (tỉ lệ mỗi mức sao); số lượt chấm trung vị mỗi người dùng `median_per_user`. Vẽ đường cong đuôi dài (số lượt chấm của phim, sắp giảm dần).

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
counts = ratings.item_id.value_counts()
top10_share = counts.iloc[: int(np.ceil(0.1 * len(counts)))].sum() / counts.sum()
rating_dist = ratings.rating.value_counts(normalize=True).sort_index()
median_per_user = ratings.groupby("user_id").size().median()
### END CODE HERE ###
plt.figure(figsize=(8, 3))
plt.plot(counts.values)
plt.xlabel("phim (sắp theo độ phổ biến)")
plt.ylabel("số lượt chấm")
plt.title("Đuôi dài")
plt.show()
print(f"10% phim phổ biến nhất chiếm {top10_share:.0%} lượt chấm | trung vị {median_per_user:.0f} lượt/người")
print("phân phối điểm:", rating_dist.round(3).to_dict())

In [ ]:
check_greater(top10_share, 0.25, "phân phối đuôi dài")
check_greater(rating_dist.loc[[4, 5]].sum(), 0.6, "điểm lệch về phía cao")

**Thiên lệch chọn lọc:** hơn 70% điểm là 4–5 sao — không phải vì phim nào cũng hay mà vì người dùng **chọn xem** phim họ đoán sẽ thích (dữ liệu thiếu không ngẫu nhiên — MNAR, chương 26). Hệ quả: (1) "chưa chấm" không có nghĩa là "không thích"; (2) đánh giá offline thiên vị phim phổ biến. Vì vậy ta coi bài toán là **xếp hạng top-N** từ tương tác (đã xem/chấm), không phải dự đoán số sao.

## Bước 2 — Chia theo thời gian

Viết `temporal_split(df, frac=0.2, min_ratings=5)`: với mỗi người dùng có ≥ `min_ratings` lượt, đưa `round(frac × n)` lượt **gần nhất** (ít nhất 1) vào phần sau; còn lại vào phần trước. Trả về `(earlier, later)`. Áp dụng → `train`, `test`; rồi áp dụng **lần nữa trên train** → `train_inner`, `valid` (để chọn siêu tham số).

In [ ]:
def temporal_split(df, frac=0.2, min_ratings=5):
    ### START CODE HERE ### (≈ 6 dòng)
    # STARTER: return df, df.iloc[:0]
    df = df.sort_values(["user_id", "timestamp"])
    rank_from_end = df.groupby("user_id").cumcount(ascending=False)
    n = df.user_id.map(df.groupby("user_id").size())
    later = (rank_from_end < (n * frac).round().clip(lower=1)) & (n >= min_ratings)
    return df[~later], df[later]
    ### END CODE HERE ###


train, test = temporal_split(ratings)
train_inner, valid = temporal_split(train)
print(f"train {len(train):,} | test {len(test):,} | train_inner {len(train_inner):,} | valid {len(valid):,}")

In [ ]:
check_equal(len(train) + len(test), len(ratings), "không mất dòng")
last_train = train.groupby("user_id").timestamp.max()
first_test = test.groupby("user_id").timestamp.min()
check_true((last_train.loc[first_test.index] <= first_test).all(), "với mỗi người dùng, test luôn sau train")
check_equal(len(test), 1082, "kích thước test")

## Bước 3 — Ma trận tương tác và hàm đánh giá

`interaction_matrix(df)`: ma trận nhị phân `N_USERS × N_ITEMS` (1 = đã chấm). Viết `evaluate(score_fn, train_df, eval_df, k=10)`: với mỗi người dùng có ít nhất một phim **thích** (điểm ≥ 4) trong `eval_df`: lấy điểm `score_fn(u)` cho mọi phim, **loại phim đã xem trong train_df**, chọn top-k; tính Precision@k, Recall@k, NDCG@k (nhị phân); trả dict trung bình + `coverage` (tỉ lệ phim khác nhau từng được gợi ý cho ít nhất một người).

In [ ]:
def interaction_matrix(df):
    M = np.zeros((N_USERS, N_ITEMS))
    M[df.user_id.values, df.item_id.values] = 1
    return M


def evaluate(score_fn, train_df, eval_df, k=10):
    ### START CODE HERE ### (≈ 16 dòng)
    # STARTER: return {"P@10": 0.0, "R@10": 0.0, "NDCG@10": 0.0, "coverage": 0.0}
    liked = eval_df[eval_df.rating >= 4].groupby("user_id").item_id.apply(set)
    seen = train_df.groupby("user_id").item_id.apply(list)
    discounts = 1 / np.log2(np.arange(2, k + 2))
    precision, recall, ndcg, recommended = [], [], [], set()
    for u, relevant in liked.items():
        scores = np.asarray(score_fn(u), dtype=float).copy()
        scores[seen.get(u, [])] = -np.inf
        top = np.argsort(-scores, kind="stable")[:k]
        recommended.update(top.tolist())
        hits = np.array([i in relevant for i in top], dtype=float)
        precision.append(hits.mean())
        recall.append(hits.sum() / len(relevant))
        ndcg.append((hits * discounts).sum() / discounts[: min(len(relevant), k)].sum())
    return {"P@10": float(np.mean(precision)), "R@10": float(np.mean(recall)), "NDCG@10": float(np.mean(ndcg)), "coverage": len(recommended) / N_ITEMS}
    ### END CODE HERE ###


toy_train = pd.DataFrame({"user_id": [0, 0], "item_id": [0, 1], "rating": [5, 5]})
toy_eval = pd.DataFrame({"user_id": [0, 0], "item_id": [2, 3], "rating": [5, 2]})
toy = evaluate(lambda u: np.r_[9.0, 8.0, 7.0, 6.0, np.zeros(N_ITEMS - 4)], toy_train, toy_eval, k=10)
print(toy)

In [ ]:
check_close(toy["P@10"], 0.1, message="1 phim thích trong top 10 (đã loại phim đã xem)")
check_close(toy["R@10"], 1.0, message="tìm được phim thích duy nhất")
check_close(toy["NDCG@10"], 1.0, message="phim thích ở vị trí đầu → NDCG = 1")
check_close(toy["coverage"], 10 / N_ITEMS, message="độ phủ")

## Bước 4 — Ba phương pháp

- `popularity`: điểm = số người đã xem phim trong train.
- `item_knn`: độ tương đồng cosine giữa các cột của ma trận tương tác $S = \frac{M^\top M}{\|m_i\|\|m_j\|}$ (đặt đường chéo = 0); điểm của người dùng $u$ = $M_u S$.
- `pure_svd(k)`: SVD rút gọn hạng $k$ của $M$ (`scipy.sparse.linalg.svds`), điểm = hàng $u$ của $U_k \Sigma_k V_k^\top$. Chọn $k \in \{5, 10, 20, 40\}$ theo NDCG@10 trên **valid** (huấn luyện trên `train_inner`) → `best_k`.

Đánh giá cả ba trên **test** (huấn luyện trên `train`) → `results` (DataFrame index tên phương pháp).

In [ ]:
def popularity_scorer(M):
    pop = M.sum(axis=0)
    return lambda u: pop


def item_knn_scorer(M):
    ### START CODE HERE ### (≈ 4 dòng)
    # STARTER: return lambda u: np.zeros(N_ITEMS)
    norms = np.linalg.norm(M, axis=0) + 1e-9
    S = (M.T @ M) / np.outer(norms, norms)
    np.fill_diagonal(S, 0)
    return lambda u: M[u] @ S
    ### END CODE HERE ###


def pure_svd_scorer(M, k):
    ### START CODE HERE ### (≈ 2 dòng)
    # STARTER: return lambda u: np.zeros(N_ITEMS)
    U, s, Vt = svds(M, k=k)
    return lambda u: (U[u] * s) @ Vt
    ### END CODE HERE ###


M_inner = interaction_matrix(train_inner)
M_train = interaction_matrix(train)
### START CODE HERE ### (≈ 4 dòng)
# STARTER: valid_ndcg, best_k, results = {}, None, None
valid_ndcg = {k: evaluate(pure_svd_scorer(M_inner, k), train_inner, valid)["NDCG@10"] for k in [5, 10, 20, 40]}
best_k = max(valid_ndcg, key=valid_ndcg.get)
results = pd.DataFrame({"phổ biến": evaluate(popularity_scorer(M_train), train, test), "item-kNN": evaluate(item_knn_scorer(M_train), train, test),
                        f"PureSVD (k={best_k})": evaluate(pure_svd_scorer(M_train, best_k), train, test)}).T
### END CODE HERE ###
print("NDCG@10 trên valid theo k:", {k: round(v, 3) for k, v in valid_ndcg.items()}, "→ chọn k =", best_k)
print(results.round(3))

In [ ]:
check_true(best_k in (5, 10), "số chiều nhỏ được chọn — SVD hạng cao overfit dữ liệu thưa")
check_greater(results.loc["item-kNN", "NDCG@10"], 1.15 * results.loc["phổ biến", "NDCG@10"], "item-kNN tốt hơn phổ biến ≥ 15%")
svd_row = results.loc[f"PureSVD (k={best_k})"]
check_greater(svd_row["NDCG@10"], results.loc["phổ biến", "NDCG@10"], "PureSVD tốt hơn phổ biến")
check_greater(svd_row["coverage"], 2 * results.loc["phổ biến", "coverage"], "PureSVD gợi ý đa dạng hơn hẳn")

## Bước 5 — Người dùng mới và thiên lệch phổ biến

**Người dùng mới** (chưa có lịch sử) không dùng được CF. Phương án: hỏi thể loại yêu thích khi đăng ký, rồi gợi ý phim phổ biến **trong thể loại đó**. Mô phỏng: với mỗi người dùng trong test, lấy thể loại xem nhiều nhất trong train làm "thể loại khai báo", điểm = (phim thuộc thể loại đó) + 0.001 × độ phổ biến → `genre_result` (dict metric trên test).

**Thiên lệch phổ biến**: với mỗi phương pháp, tính độ phổ biến trung bình (phân vị, 0–1) của các phim được gợi ý → `popularity_bias` (Series).

In [ ]:
genre_of = items.set_index("item_id").genre.reindex(range(N_ITEMS))
popularity_rank = pd.Series(M_train.sum(axis=0)).rank(pct=True).values
### START CODE HERE ### (≈ 12 dòng)
# STARTER: genre_result, popularity_bias = None, None
favorite_genre = train.assign(genre=train.item_id.map(genre_of)).groupby("user_id").genre.agg(lambda g: g.value_counts().index[0])
pop = M_train.sum(axis=0)
genre_result = evaluate(lambda u: (genre_of.values == favorite_genre.get(u)).astype(float) + 1e-3 * pop, train, test)
scorers = {"phổ biến": popularity_scorer(M_train), "item-kNN": item_knn_scorer(M_train), f"PureSVD (k={best_k})": pure_svd_scorer(M_train, best_k)}
seen = train.groupby("user_id").item_id.apply(list)
bias = {}
for name, fn in scorers.items():
    tops = []
    for u in test.user_id.unique():
        s = np.asarray(fn(u), dtype=float).copy()
        s[seen.get(u, [])] = -np.inf
        tops.extend(np.argsort(-s, kind="stable")[:10])
    bias[name] = popularity_rank[tops].mean()
popularity_bias = pd.Series(bias)
### END CODE HERE ###
print("theo thể loại khai báo:", {k: round(v, 3) for k, v in genre_result.items()})
print("phân vị phổ biến TB của phim được gợi ý:", popularity_bias.round(3).to_dict())

In [ ]:
check_greater(genre_result["NDCG@10"], 0.5 * results.loc["phổ biến", "NDCG@10"], "gợi ý theo thể loại dùng được cho người mới")
check_greater(genre_result["coverage"], results.loc["phổ biến", "coverage"], "đa dạng hơn danh sách phổ biến chung")
check_greater(popularity_bias["phổ biến"], popularity_bias[f"PureSVD (k={best_k})"], "PureSVD ít thiên lệch phổ biến hơn")

## Bước 6 — Báo cáo

Viết 5–8 câu cho trưởng nhóm sản phẩm: phương pháp đề xuất, cải thiện offline so với trang chủ hiện tại, đánh đổi độ chính xác ↔ đa dạng, chiến lược cho người dùng mới, giới hạn của đánh giá offline (thiên lệch chọn lọc), đề xuất A/B test (chương 40) với metric online (tỉ lệ bấm xem, thời gian xem).

✍️ Báo cáo:

**Báo cáo mẫu:** Gợi ý cá nhân hoá bằng lọc cộng tác (item-kNN hoặc PureSVD hạng thấp) tăng NDCG@10 offline 25–40% so với trang chủ "phim phổ biến" hiện tại (PureSVD hạng 5 tốt nhất), và PureSVD còn gợi ý đa dạng hơn hẳn (phủ ~38% danh mục thay vì ~16%), giảm thiên lệch về vài phim ăn khách. Số chiều SVD phải nhỏ (validation chọn k = 5) — mô hình phức tạp hơn overfit vì dữ liệu rất thưa (trung vị 15 lượt/người). Người dùng mới: hỏi thể loại yêu thích khi đăng ký và gợi ý phim nổi bật trong thể loại đó cho đến khi có ~5 lượt xem. Lưu ý: đánh giá offline dựa trên những phim người dùng **đã tự chọn xem** (thiên lệch chọn lọc) nên có xu hướng ưu ái phim phổ biến; cần A/B test với tỉ lệ bấm xem và thời gian xem làm metric chính, kèm một phần nhỏ gợi ý khám phá ngẫu nhiên để thu dữ liệu không thiên lệch (vòng phản hồi — chương 40).